In [1]:
from datetime import date
import pandas as pd
import numpy as np
import utils.fetch_dbs
import json
import mysql.connector

pd.set_option('display.max_columns', None)

with open('utils/secrets.json', 'r') as f:
    secrets = json.load(f)

In [166]:
import importlib
importlib.reload(utils.fetch_dbs)

<module 'utils.fetch_dbs' from '/nfs/cellgeni/tickets/tic-4679/actions/hdca_v2_reprocessing/utils/fetch_dbs.py'>

In [2]:
datasets = pd.read_csv('hdca_reprocessing - accessions.csv')
datasets

,publication_id,accession,institute,fastq avialability,source,status,done by,secondary publications,comment,link,old_title,key
0,calvanese_2022_agm,GSE162950,"Broad Stem Cell Research Center, UCLA",open,external,done,alex,NaN,"we use just four samples, I guess AGM stands f...",NaN,Calvanese_et_al_2022_nature,NaN
1,calvanese_2022_agm,GSE135202,"Broad Stem Cell Research Center, UCLA",open,external,incomplete,alex,NaN,NaN,NaN,NaN,NaN
2,dong_2020_adrenal,GSE137804,Children's Hospital of Fudan University,open,external,done,pasha,NaN,"just four samples, to be selected by names",NaN,Dong_et_al_2022_CancerCell,NaN
3,braun_2023_brain,EGAD50000001295,Karolinska Institutet,protected,external,done,alex,NaN,EGAS00001004107,NaN,Braun_et_al._2022_bioRxiv,NaN
4,braun_2023_brain,EGAD00001006049,Karolinska Institutet,protected,external,done,alex,NaN,EGAS00001004107,NaN,Braun_et_al._2022_bioRxiv,NaN
5,collin_2021_cornea_conjunctiva,GSE155683,Newcastle University,open,external,done,pasha,NaN,NaN,NaN,Colin_et_al_2022_ScienceDirect,NaN
6,yu_2021_stomach,E-MTAB-10187,Roche,open,external,done,alex,NaN,NaN,NaN,Yu_et_al_2020_Cell,NaN
7,miller_2020_lung_airway_trachea,E-MTAB-8221,University of Michigan,open,external,done,alex,NaN,NaN,NaN,Miller_et_al_2020_Developmental_Cell,NaN
8,sridhar_2020_retina,GSE142526,University of Washington,open,external,done,pasha,NaN,NaN,NaN,Sridhar_et_al_2020_CellPress,NaN
9,garcia_alonso_2022_gonads,E-MTAB-10551,Wellcome Sanger Institute,open,external,done,alex,NaN,NaN,NaN,Garcia_Alonso_et_al_2022_nature,NaN


In [3]:
datasets.accession.value_counts()

accession
GSE162950                          1
GSE135202                          1
GSE137804                          1
EGAD50000001295                    1
EGAD00001006049                    1
GSE155683                          1
E-MTAB-10187                       1
E-MTAB-8221                        1
GSE142526                          1
E-MTAB-10551                       1
E-MTAB-11708                       1
EGAD00001009386                    1
Garcia_Alonso_et_al_2022_nature    1
E-MTAB-8901                        1
Kanemaru_et_al_2024_bioRxiv        1
EGAD00001009801                    1
E-MTAB-9536                        1
E-MTAB-9389                        1
PRJEB37165                         1
E-MTAB-11343                       1
E-MTAB-9801                        1
PRJEB41514                         1
E-MTAB-11504                       1
E-MTAB-11520                       1
E-MTAB-11911                       1
E-MTAB-8813                        1
GSE260715                   

# Collect metadata from databases
## ArrayExpress

In [4]:
inx = datasets.accession.str.startswith('E-MTAB')
inx = inx[inx].index
for i in inx:
    a = datasets.accession[i]
    print(a)
    url = 'https://www.ebi.ac.uk/biostudies/files/'+a+'/'+a+'.sdrf.txt'
    if pd.notna(datasets.key[i]):
        url = url+"?key="+datasets.key[i]
    d = pd.read_csv(url, sep="\t")
    d.to_csv("datasets_metadata/"+a+".csv")

E-MTAB-10187
E-MTAB-8221
E-MTAB-10551
E-MTAB-11708
E-MTAB-8901
E-MTAB-9536
E-MTAB-9389
E-MTAB-11343
E-MTAB-9801
E-MTAB-11504
E-MTAB-11520
E-MTAB-11911
E-MTAB-8813
E-MTAB-6701
E-MTAB-14385
E-MTAB-11278
E-MTAB-7407
E-MTAB-13071
E-MTAB-10552
E-MTAB-11673
E-MTAB-8581


## EGA

In [6]:
f = datasets.accession.str.startswith('EGAD')
for a in datasets.accession[f]:
    print(a)
    d = utils.fetch_dbs.get_EGAD_files(a)
    d.to_csv("datasets_metadata/"+a+".csv")

EGAD50000001295
EGAD00001006049
EGAD00001009386
EGAD00001009801
EGAD00001015384


## GEO

In [7]:
f = datasets.accession.str.startswith('GSE')
for a in datasets.accession[f]:
    print(a)
    url = 'https://ftp.ncbi.nlm.nih.gov/geo/series/'+a[:6]+'nnn/'+a+'/soft/'+a+'_family.soft.gz'
    d = utils.fetch_dbs.get_geo_family_soft_samples(url)
    d.to_csv("datasets_metadata/"+a+".csv")

GSE162950
GSE135202
GSE137804
GSE155683
GSE142526
GSE260715
GSE157329
GSE171892
GSE264407
GSE215895
GSE271304
GSE147520
GSE159745
GSE245310
GSE201586


## ENA

In [8]:
f = datasets.accession.str.startswith('PRJEB')
for a in datasets.accession[f]:
    print(a)
    url = 'https://www.ebi.ac.uk/ena/portal/api/filereport?accession='+a+'&result=read_run&fields=study_accession,experiment_accession,sample_accession,secondary_sample_accession,sample_title&format=tsv'
    d = pd.read_csv(url, sep="\t")
    d.to_csv("datasets_metadata/"+a+".csv")

PRJEB37165
PRJEB41514
PRJEB77091


## CNCB
didn't find working API, anyway we need to get access first

# Concatenate metadata
The goal is to get list of samples with minimal info:

1. dataset_acc - primary id
2. library_id - primary id
3. author_sample_id - the name used in object (hopefully)
4. irods_name - how we identify it locally
5. [library_type]

## Array Express

In [3]:
f = datasets.accession.str.startswith('E-MTAB')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    # hooks for non-usual datasets
    if a == 'E-MTAB-11278':
        d['Source Name'] = d['Derived Array Data File'].str.replace('.tar.gz','')
    if a == 'E-MTAB-8813':
        d['Source Name'] = d['Derived Array Data File.1'].str.replace('.tar.gz','')
    if a == 'E-MTAB-8901':
        d['Source Name'] = [x[0] for x in d['Comment[read1 file]'].str.split('_')]
    if a == 'E-MTAB-8581':
        d['Source Name'] = d['Comment[original source name]']
    dss.append(d)

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

ae = pd.concat([df[common_cols] for df in dss], ignore_index=True)
ae[:3]

,Source Name,Comment[ENA_RUN],Extract Name,Comment[umi barcode offset],Unnamed: 0,Comment[primer],Characteristics[individual],Comment[umi barcode read],Comment[LIBRARY_SOURCE],Scan Name,Comment[input molecule],Comment[single cell isolation],Characteristics[developmental stage],Comment[cell barcode offset],Technology Type,Comment[ENA_EXPERIMENT],Comment[cell barcode read],Protocol REF,dataset_acc,Protocol REF.3,Protocol REF.1,Comment[LIBRARY_STRATEGY],Characteristics[organism],Comment[LIBRARY_LAYOUT],Comment[FASTQ_URI],Assay Name,Comment[LIBRARY_SELECTION],Comment[BioSD_SAMPLE],Comment[umi barcode size],Comment[library construction],Comment[cell barcode size],Comment[end bias],Protocol REF.2,Material Type,Performer,Comment[ENA_SAMPLE]
0,H9-Definitive-endoderm,ERR5417814,H9-Definitive-endoderm,16.0,0,oligo-dT,10,read 1,TRANSCRIPTOMIC SINGLE CELL,H9-DE_S1_L001.bam,polyA RNA,10x,not available,0.0,sequencing assay,ERX5202382,read 1,P-MTAB-107118,E-MTAB-10187,P-MTAB-107113,P-MTAB-107111,RNA-Seq,Homo sapiens,SINGLE,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,H9-DE_S1_L001,Oligo-dT,SAMEA8203536,10.0,10xV2,16.0,3 prime tag,P-MTAB-107112,organoid,University of Michigan Advanced Genomics Core,ERS5890137
1,H9-Definitive-endoderm,ERR5417815,H9-Definitive-endoderm,16.0,1,oligo-dT,10,read 1,TRANSCRIPTOMIC SINGLE CELL,H9-DE_S1_L002.bam,polyA RNA,10x,not available,0.0,sequencing assay,ERX5202382,read 1,P-MTAB-107118,E-MTAB-10187,P-MTAB-107113,P-MTAB-107111,RNA-Seq,Homo sapiens,SINGLE,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,H9-DE_S1_L002,Oligo-dT,SAMEA8203536,10.0,10xV2,16.0,3 prime tag,P-MTAB-107112,organoid,University of Michigan Advanced Genomics Core,ERS5890137
2,H9-Definitive-endoderm,ERR5417816,H9-Definitive-endoderm,16.0,2,oligo-dT,10,read 1,TRANSCRIPTOMIC SINGLE CELL,H9-DE_S1_L003.bam,polyA RNA,10x,not available,0.0,sequencing assay,ERX5202382,read 1,P-MTAB-107118,E-MTAB-10187,P-MTAB-107113,P-MTAB-107111,RNA-Seq,Homo sapiens,SINGLE,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,H9-DE_S1_L003,Oligo-dT,SAMEA8203536,10.0,10xV2,16.0,3 prime tag,P-MTAB-107112,organoid,University of Michigan Advanced Genomics Core,ERS5890137


### Remove unwanted samples

In [4]:
ae.loc[ae['Comment[ENA_RUN]'].isna(),'dataset_acc'].value_counts() # samples with no raw data on ArrayExpress, remove them

dataset_acc
E-MTAB-10551    28
E-MTAB-11708     4
Name: count, dtype: int64

In [5]:
ae = ae[~ae['Comment[ENA_RUN]'].isna()]

In [6]:
ae['Comment[LIBRARY_STRATEGY]'].value_counts()

Comment[LIBRARY_STRATEGY]
RNA-Seq     1665
OTHER        347
ATAC-seq     216
Name: count, dtype: int64

In [7]:
ae.loc[ae['Comment[LIBRARY_STRATEGY]']=='ATAC-seq','Comment[library construction]'].value_counts()

Comment[library construction]
scATAC-seq    180
other          36
Name: count, dtype: int64

In [8]:
ae.loc[ae['Comment[LIBRARY_STRATEGY]']=='RNA-Seq','Comment[library construction]'].value_counts()

Comment[library construction]
10xV2        464
10x 5' v2    384
10x 5' v1    287
10xV3        153
10x 3' v3    144
10xV1        140
10x 3' v2     66
10x 3’ v2     27
Name: count, dtype: int64

In [9]:
ae.loc[ae['Comment[LIBRARY_STRATEGY]']=='OTHER','Comment[library construction]'].value_counts()

Comment[library construction]
10x 5' v1             116
10x 3' v2              93
10x TCR enrichment     87
10x BCR enrichment     48
10x 3' v3               3
Name: count, dtype: int64

In [10]:
ae = ae.loc[ae['Comment[LIBRARY_STRATEGY]']!='ATAC-seq',]
ae = ae.loc[ae['Comment[library construction]']!='10x TCR enrichment',]
ae = ae.loc[ae['Comment[library construction]']!='10x BCR enrichment',]

In [11]:
ae['Comment[library construction]'].value_counts()

Comment[library construction]
10xV2        464
10x 5' v1    403
10x 5' v2    384
10x 3' v2    159
10xV3        153
10x 3' v3    147
10xV1        140
10x 3’ v2     27
Name: count, dtype: int64

In [12]:
ae['Material Type'].value_counts()

Material Type
cell             1088
organism part     739
organoid           50
Name: count, dtype: int64

In [13]:
ae.loc[ae['Material Type']=='organoid','dataset_acc'].value_counts()

dataset_acc
E-MTAB-10187    32
E-MTAB-8221     18
Name: count, dtype: int64

In [14]:
ae['Characteristics[organism]'].value_counts()

Characteristics[organism]
Homo sapiens    1865
mixed sample       6
Mus musculus       6
Name: count, dtype: int64

In [15]:
ae = ae[ae['Characteristics[organism]']=='Homo sapiens']

In [16]:
ae = ae.loc[ae['Material Type']!='organoid',['dataset_acc','Comment[ENA_SAMPLE]','Source Name','Comment[library construction]']].drop_duplicates()

In [17]:
ae

,dataset_acc,Comment[ENA_SAMPLE],Source Name,Comment[library construction]
32,E-MTAB-10187,ERS5890146,HT-185_Duodenum,10xV2
35,E-MTAB-10187,ERS5890147,HT-188_Duodenum,10xV2
41,E-MTAB-10187,ERS5890148,HT-228_Colon,10xV2
49,E-MTAB-10187,ERS5890149,HT-228_Ileum,10xV2
57,E-MTAB-10187,ERS5890150,HT-228_Liver,10xV2
...,...,...,...,...
2255,E-MTAB-8581,ERS4228604,WSSS8062675,10xV2
2256,E-MTAB-8581,ERS4228662,WSSS8084742,10xV3
2257,E-MTAB-8581,ERS4228663,WSSS8084743,10xV3
2258,E-MTAB-8581,ERS4228664,WSSS8084744,10xV3


In [18]:
# keep only GEX Sanger id in author_sample_id for To_et_al_2024_nature
f = ae.dataset_acc=='E-MTAB-14385'
ae.loc[f,'Source Name'] = [x[0] for x in ae.loc[f,'Source Name'].str.split('_and_')]

## EGA

In [19]:
f = datasets.accession.str.startswith('EGAD')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    dss.append(d)

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

ega = pd.concat([df[common_cols] for df in dss], ignore_index=True)
ega = ega[['dataset_acc','EGAN','AUTHOR_ID','LIBRARY_TYPE']].drop_duplicates()
ega[:3]

,dataset_acc,EGAN,AUTHOR_ID,LIBRARY_TYPE
0,EGAD50000001295,EGAN50000223075,10X147_1,10Xchromium 3' single cell version 2
1,EGAD50000001295,EGAN50000223066,10X186_1,10Xchromium 3' single cell version 2
2,EGAD50000001295,EGAN50000223076,10X186_4,10Xchromium 3' single cell version 2


In [20]:
pd.crosstab(ega.LIBRARY_TYPE,ega.dataset_acc)

dataset_acc,EGAD00001009386,EGAD00001009801,EGAD00001015384,EGAD50000001295
LIBRARY_TYPE,,,,
10Xchromium 3' single cell version 2,0,0,0,69
Chromium Visium,0,7,7,0
Chromium single cell 3 prime v3,0,52,0,0
Chromium single cell 5 prime,28,0,7,0
Chromium single cell TCR,0,0,5,0


In [21]:
ega = ega[~ega.LIBRARY_TYPE.isin(['Chromium single cell TCR','Chromium Visium'])]
pd.crosstab(ega.LIBRARY_TYPE,ega.dataset_acc)

dataset_acc,EGAD00001009386,EGAD00001009801,EGAD00001015384,EGAD50000001295
LIBRARY_TYPE,,,,
10Xchromium 3' single cell version 2,0,0,0,69
Chromium single cell 3 prime v3,0,52,0,0
Chromium single cell 5 prime,28,0,7,0


## GEO

In [22]:
f = datasets.accession.str.startswith('GSE')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    # hooks for non-usual datasets
    if a == 'GSE162950':
        d = d[~d['title'].str.startswith('Spatial_transcriptomics')]
        d = d[~d['title'].str.startswith('EB-')] # looks like cell line
        
    dss.append(d)
        

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

geo = pd.concat([df[common_cols] for df in dss], ignore_index=True)
geo[:3]

,submission_date,type,Unnamed: 0,contact_zip/postal_code,contact_address,library_strategy,library_source,contact_name,characteristics_ch1,contact_country,title,dataset_acc,data_processing,supplementary_file_1,source_name_ch1,data_row_count,channel_count,last_update_date,platform_id,contact_institute,instrument_model,status,organism_ch1,sample_id,molecule_ch1,extract_protocol_ch1,series_id,taxid_ch1,geo_accession,contact_city,library_selection
0,Dec 09 2020,SRA,0,90095,"610, Charles Young Dr East, TLSB 3000C",RNA-Seq,transcriptomic,"Feiyang,,Ma",time: 4wk,USA,aorta-gonad-mesonephros 4.5 week,GSE162950,Sequencing was performed on Illumina NovaSeq 6...,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4968...,aorta-gonad-mesonephros,0,1,Jan 20 2022,GPL24676,UCLA,Illumina NovaSeq 6000,Public on Jan 20 2022,Homo sapiens,GSM4968831,total RNA,"For single cell RNA-sequencing, dissociated ce...",GSE162950,9606,GSM4968831,Los Angeles,cDNA
1,Dec 09 2020,SRA,1,90095,"610, Charles Young Dr East, TLSB 3000C",RNA-Seq,transcriptomic,"Feiyang,,Ma",time: 5wk,USA,aorta-gonad-mesonephros 5 week,GSE162950,Sequencing was performed on Illumina NovaSeq 6...,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4968...,aorta-gonad-mesonephros,0,1,Jan 20 2022,GPL24676,UCLA,Illumina NovaSeq 6000,Public on Jan 20 2022,Homo sapiens,GSM4968832,total RNA,"For single cell RNA-sequencing, dissociated ce...",GSE162950,9606,GSM4968832,Los Angeles,cDNA
2,Dec 09 2020,SRA,2,90095,"610, Charles Young Dr East, TLSB 3000C",RNA-Seq,transcriptomic,"Feiyang,,Ma",time: 5wk,USA,aorta-gonad-mesonephros 5.5 week,GSE162950,Sequencing was performed on Illumina NovaSeq 6...,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4968...,aorta-gonad-mesonephros,0,1,Jan 20 2022,GPL24676,UCLA,Illumina NovaSeq 6000,Public on Jan 20 2022,Homo sapiens,GSM4968833,total RNA,"For single cell RNA-sequencing, dissociated ce...",GSE162950,9606,GSM4968833,Los Angeles,cDNA


In [23]:
geo['molecule_ch1'].value_counts()

molecule_ch1
polyA RNA      239
total RNA      153
protein         11
other           11
genomic DNA      4
Name: count, dtype: int64

In [24]:
geo['library_source'].value_counts()

library_source
transcriptomic                304
transcriptomic single cell     88
other                          22
genomic                         4
Name: count, dtype: int64

In [25]:
geo['library_strategy'].value_counts()

library_strategy
RNA-Seq     381
OTHER        33
ATAC-seq      4
Name: count, dtype: int64

In [26]:
geo['library_selection'].value_counts()

library_selection
cDNA     381
other     37
Name: count, dtype: int64

In [27]:
pd.crosstab(geo['molecule_ch1'],geo['library_strategy'])

library_strategy,ATAC-seq,OTHER,RNA-Seq
molecule_ch1,,,
genomic DNA,4,0,0
other,0,11,0
polyA RNA,0,0,239
protein,0,11,0
total RNA,0,11,142


In [28]:
# CITE, do not need it
pd.set_option('display.max_rows', 20)
geo.loc[geo['molecule_ch1']=='other','title']

310     TT-CITE-1-TCRab
311     TT-CITE-2-TCRab
312     TT-CITE-3-TCRab
313     TT-CITE-4-TCRab
314     TT-CITE-5-TCRab
315     TT-CITE-6-TCRab
316     TT-CITE-8-TCRab
317     TT-CITE-9-TCRab
318    TT-CITE-10-TCRab
319    TT-CITE-11-TCRab
320    TT-CITE-12-TCRab
Name: title, dtype: object

In [29]:
geo = geo.loc[(geo['library_strategy'] == 'RNA-Seq') | (geo['molecule_ch1'].isin(['polyA RNA','total RNA'])),]
pd.crosstab(geo['molecule_ch1'],geo['library_strategy'])

library_strategy,OTHER,RNA-Seq
molecule_ch1,,
polyA RNA,0,239
total RNA,11,142


In [30]:
geo=geo.loc[:,['dataset_acc','geo_accession','title']].drop_duplicates()
geo['library_type'] = pd.NA
geo

,dataset_acc,geo_accession,title,library_type
0,GSE162950,GSM4968831,aorta-gonad-mesonephros 4.5 week,<NA>
1,GSE162950,GSM4968832,aorta-gonad-mesonephros 5 week,<NA>
2,GSE162950,GSM4968833,aorta-gonad-mesonephros 5.5 week,<NA>
3,GSE162950,GSM4968834,aorta-gonad-mesonephros 6 week,<NA>
4,GSE162950,GSM4968835,cord 4.5 week,<NA>
...,...,...,...,...
413,GSE201586,GSM6068106,humanDRG_sample14,<NA>
414,GSE201586,GSM6068107,humanDRG_sample15,<NA>
415,GSE201586,GSM6068108,humanDRG_sample16,<NA>
416,GSE201586,GSM6068109,humanDRG_sample17,<NA>


## ENA

In [31]:
f = datasets.accession.str.startswith('PRJEB')
dss = []
for a in datasets.accession[f]:
    d = pd.read_csv("datasets_metadata/"+a+".csv")
    d['dataset_acc'] = a
    dss.append(d)
    # hooks for non-usual datasets
    if a =='PRJEB37165':
        d['sample_title'] = [x[0] for x in d['sample_title'].str.split('_')]
    if a =='PRJEB41514': # wrong sample_titles, lets get them from Sanger db
        d['sample_title'] = [utils.fetch_dbs.fetch_mlw_query(secrets['mlw'],"SELECT name FROM sample WHERE accession_number = '"+ers+"'").iloc[0,0] for ers in d['secondary_sample_accession']]

common_cols = list(set.intersection(*(set(df.columns) for df in dss)))

ena = pd.concat([df[common_cols] for df in dss], ignore_index=True)
ena = ena.loc[:,['dataset_acc','secondary_sample_accession','sample_title']].drop_duplicates()
ena['library_type'] = pd.NA
ena

/nfs/cellgeni/tickets/tic-4679/actions/hdca_v2_reprocessing/utils/fetch_dbs.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, db)


,dataset_acc,secondary_sample_accession,sample_title,library_type
0,PRJEB37165,ERS4605096,CZIKidney7587406,<NA>
1,PRJEB37165,ERS4605083,FCAImmP7555850,<NA>
2,PRJEB37165,ERS4605078,CZIKidney7587405,<NA>
3,PRJEB37165,ERS4605079,4834STDY7002885,<NA>
4,PRJEB37165,ERS4605094,FCAImmP7528293,<NA>
...,...,...,...,...
143,PRJEB77091,ERS20592939,TA11486161,<NA>
144,PRJEB77091,ERS20592941,TA11486163,<NA>
145,PRJEB77091,ERS20592942,TA11486164,<NA>
146,PRJEB77091,ERS20592946,TA11556492,<NA>


## Internal

In [32]:
we_11pcv = pd.read_csv('datasets_metadata/whole_embryo_11pcw.csv')
we_11pcv['dataset_acc'] = 'whole_embryo_11pcw'
we_11pcv['library_type'] = pd.NA
we_11pcv = we_11pcv[['dataset_acc','Sanger ID','Sanger ID','library_type']].drop_duplicates()
we_11pcv

,dataset_acc,Sanger ID,Sanger ID,library_type
0,whole_embryo_11pcw,HDBI_wEMB14781754,HDBI_wEMB14781754,<NA>
1,whole_embryo_11pcw,HDBI_wEMB14781755,HDBI_wEMB14781755,<NA>
2,whole_embryo_11pcw,HDBI_wEMB14781756,HDBI_wEMB14781756,<NA>
3,whole_embryo_11pcw,HDBI_wEMB14781757,HDBI_wEMB14781757,<NA>
4,whole_embryo_11pcw,HDBI_wEMB14781758,HDBI_wEMB14781758,<NA>
...,...,...,...,...
43,whole_embryo_11pcw,HDBI_wEMB14781797,HDBI_wEMB14781797,<NA>
44,whole_embryo_11pcw,HDBI_wEMB14781798,HDBI_wEMB14781798,<NA>
45,whole_embryo_11pcw,HDBI_wEMB14781799,HDBI_wEMB14781799,<NA>
46,whole_embryo_11pcw,HDBI_wEMB14781800,HDBI_wEMB14781800,<NA>


In [33]:
internal = pd.read_csv('datasets_metadata/internal_datasets.csv')
internal['library_type'] = pd.NA
internal['dataset_acc'] = internal['dataset']
internal = internal[['dataset_acc','sample_title','sample_title','library_type']]
internal

,dataset_acc,sample_title,sample_title,library_type
0,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea10402917,BHF_F_Hea10402917,<NA>
1,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea10402918,BHF_F_Hea10402918,<NA>
2,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11064670,BHF_F_Hea11064670,<NA>
3,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11064671,BHF_F_Hea11064671,<NA>
4,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11064672,BHF_F_Hea11064672,<NA>
...,...,...,...,...
36,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828900,HCA_F_GON10828900,<NA>
37,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828901,HCA_F_GON10828901,<NA>
38,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828902,HCA_F_GON10828902,<NA>
39,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10941968,HCA_F_GON10941968,<NA>


## Merge

In [34]:
ae.columns = geo.columns = ena.columns = we_11pcv.columns = internal.columns  = ega.columns = ['dataset_acc','library_id','author_sample_id','library_type']

In [35]:
samples = pd.concat([ae,ega,geo,ena,we_11pcv,internal], ignore_index=True)
samples['institute'] = samples.dataset_acc.replace(dict(zip(datasets.accession,datasets.institute)))
samples['source'] = samples.dataset_acc.replace(dict(zip(datasets.accession,datasets.source)))
samples['publication_id'] = samples.dataset_acc.replace(dict(zip(datasets.accession,datasets.publication_id)))
samples

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id
0,E-MTAB-10187,ERS5890146,HT-185_Duodenum,10xV2,Roche,external,yu_2021_stomach
1,E-MTAB-10187,ERS5890147,HT-188_Duodenum,10xV2,Roche,external,yu_2021_stomach
2,E-MTAB-10187,ERS5890148,HT-228_Colon,10xV2,Roche,external,yu_2021_stomach
3,E-MTAB-10187,ERS5890149,HT-228_Ileum,10xV2,Roche,external,yu_2021_stomach
4,E-MTAB-10187,ERS5890150,HT-228_Liver,10xV2,Roche,external,yu_2021_stomach
...,...,...,...,...,...,...,...
1877,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828900,HCA_F_GON10828900,NaN,Wellcome Sanger Institute,internal,garcia_alonso_2022_gonads
1878,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828901,HCA_F_GON10828901,NaN,Wellcome Sanger Institute,internal,garcia_alonso_2022_gonads
1879,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10828902,HCA_F_GON10828902,NaN,Wellcome Sanger Institute,internal,garcia_alonso_2022_gonads
1880,Garcia_Alonso_et_al_2022_nature,HCA_F_GON10941968,HCA_F_GON10941968,NaN,Wellcome Sanger Institute,internal,garcia_alonso_2022_gonads


# Check completeness

In [36]:
set(datasets.accession) - set(samples.dataset_acc)

{'HRA002757', 'HRA005835', 'OMIX003147', 'phs002003.v1.p1'}

In [37]:
# tmp column to use to point to processed data on irods. 
# EGA samples are named by author_sample_id instead of sample_acc
samples['irods_name'] = samples['library_id']

# braun_2023_brain is named by modified (see below) author id
f = samples['publication_id'] == 'braun_2023_brain'
samples.loc[f,'irods_name'] = samples.loc[f,'author_sample_id']

# all internal samples are named by Sanger id on irods
f = samples['source'] == 'internal'
samples.loc[f,'irods_name'] = samples.loc[f,'author_sample_id']


In [38]:
# look for samples found in multiple submissions
t=samples.author_sample_id.value_counts()
t = [[x,', '.join(sorted(samples.publication_id[samples.author_sample_id==x].tolist())),', '.join(sorted(samples.dataset_acc[samples.author_sample_id==x].tolist()))]  for x in t[t>1].index]
t=pd.DataFrame(t,columns=['author_sample_id','studies','datasets'])
t.sort_values('datasets')


,author_sample_id,studies,datasets
0,FCAImmP7862093,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
9,FCAImmP7862085,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
7,FCAImmP7862087,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
6,FCAImmP7862088,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
5,FCAImmP7862089,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
8,FCAImmP7862086,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
3,FCAImmP7862091,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
2,FCAImmP7862092,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
1,FCAImmP7862084,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"
4,FCAImmP7862090,"goh_2023_yolksac, suo_2022_immune_integration","E-MTAB-10552, E-MTAB-11343"


In [39]:
# keep first dataset if sample found in two datasets
tmp = samples[samples.dataset_acc==datasets.accession[0]]
for a in datasets.accession[1:]:
    tmp = pd.concat([tmp,samples[(samples.dataset_acc==a) & ~(samples.author_sample_id.isin(tmp.author_sample_id))]])
samples = tmp
samples.index = samples.library_id

In [40]:
samples

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name
library_id,,,,,,,,
GSM4968831,GSE162950,GSM4968831,aorta-gonad-mesonephros 4.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968831
GSM4968832,GSE162950,GSM4968832,aorta-gonad-mesonephros 5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968832
GSM4968833,GSE162950,GSM4968833,aorta-gonad-mesonephros 5.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968833
GSM4968834,GSE162950,GSM4968834,aorta-gonad-mesonephros 6 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968834
GSM4968835,GSE162950,GSM4968835,cord 4.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968835
...,...,...,...,...,...,...,...,...
GSM6068106,GSE201586,GSM6068106,humanDRG_sample14,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068106
GSM6068107,GSE201586,GSM6068107,humanDRG_sample15,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068107
GSM6068108,GSE201586,GSM6068108,humanDRG_sample16,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068108


## Check samples from HDCA v1

In [41]:
samples_v1 = pd.read_csv('libraries_lists/libraries_v1.csv')
samples_v1

,institute,study,library_id_fixed,n_cells,library_id_is_sanger,n_unique_bs,dataset_acc,library_acc,Unnamed: 8,comment
0,Karolinska,Braun_et_al._2022_bioRxiv,10X101_1,6610,False,6610,EGAD00001006049,10X101_1,NaN,NaN
1,Karolinska,Braun_et_al._2022_bioRxiv,10X101_2,7579,False,7579,EGAD00001006049,10X101_2,NaN,NaN
2,Karolinska,Braun_et_al._2022_bioRxiv,10X101_3,7635,False,7635,EGAD00001006049,10X101_3,NaN,NaN
3,Karolinska,Braun_et_al._2022_bioRxiv,10X101_4,6970,False,6970,EGAD00001006049,10X101_4,NaN,NaN
4,Karolinska,Braun_et_al._2022_bioRxiv,10X101_5,6135,False,6135,EGAD00001006049,10X101_5,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
975,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384954,3557,True,3557,Sanger,WSSS_THYst9384954,NaN,NaN
976,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384955,3917,True,3917,Sanger,WSSS_THYst9384955,NaN,NaN
977,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384956,4862,True,4862,Sanger,WSSS_THYst9384956,NaN,NaN
978,Wellcome_Sanger,Zhang_et_al_2020_bioRxiv,WSSS_THYst9384957,3624,True,3624,Sanger,WSSS_THYst9384957,NaN,NaN


In [42]:
samples_v1[~(samples_v1.library_id_fixed.isin(samples.author_sample_id))]

,institute,study,library_id_fixed,n_cells,library_id_is_sanger,n_unique_bs,dataset_acc,library_acc,Unnamed: 8,comment
340,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,AGM_555,1467,False,1467,GSE162950,GSM4968832,NaN,based on names of supp files
341,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,AGM_563,3228,False,3228,GSE162950,GSM4968834,NaN,based on names of supp files
342,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,AGM_575,4108,False,4108,GSE162950,GSM4968833,NaN,based on names of supp files
343,Broad_Stem_Cell_Research_centre,Calvanese_et_al_2022_nature,AGM_658,5276,False,5276,GSE162950,GSM4968831,NaN,based on names of supp files
344,Newcastle_University,Colin_et_al_2022_ScienceDirect,F1014615,1082,False,1082,GSE155683,GSM4710450,NaN,based on last 5 digits
...,...,...,...,...,...,...,...,...,...,...
950,Roche,Yu_et_al_2020_Cell,HT-231_D117_Stomach-antrum,1160,False,1160,E-MTAB-10187,ERS5890152,NaN,HT-231_Stomach-antrum
951,Roche,Yu_et_al_2020_Cell,HT-231_D117_Stomach-corpus,1516,False,1516,E-MTAB-10187,ERS5890153,NaN,HT-231_Stomach-corpus
952,Roche,Yu_et_al_2020_Cell,HT-232_D100_Stomach-antrum,2156,False,2156,E-MTAB-10187,ERS5890155,NaN,HT-232_Stomach-antrum
953,Roche,Yu_et_al_2020_Cell,HT-232_D100_Stomach-corpus,2639,False,2639,E-MTAB-10187,ERS5890156,NaN,HT-232_Stomach-corpus


In [43]:
# not sure where the _ABC_1/AB_ in Braun are coming from...

In [44]:
samples_v1_braun = samples_v1[samples_v1.study == 'Braun_et_al._2022_bioRxiv']
samples_v1_braun = dict(zip(samples_v1_braun.library_id_fixed,samples_v1_braun.library_acc))
samples.irods_name = samples.irods_name.replace(samples_v1_braun)

In [45]:
samples[samples.publication_id == 'braun_2023_brain']

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name
library_id,,,,,,,,
EGAN50000223075,EGAD50000001295,EGAN50000223075,10X147_1,10Xchromium 3' single cell version 2,Karolinska Institutet,external,braun_2023_brain,10X147_1_ABC_1
EGAN50000223066,EGAD50000001295,EGAN50000223066,10X186_1,10Xchromium 3' single cell version 2,Karolinska Institutet,external,braun_2023_brain,10X186_1_ABC_1
EGAN50000223076,EGAD50000001295,EGAN50000223076,10X186_4,10Xchromium 3' single cell version 2,Karolinska Institutet,external,braun_2023_brain,10X186_4_ABC_1
EGAN50000223131,EGAD50000001295,EGAN50000223131,10X187_1,10Xchromium 3' single cell version 2,Karolinska Institutet,external,braun_2023_brain,10X187_1_ABC_1
EGAN50000223072,EGAD50000001295,EGAN50000223072,10X187_2,10Xchromium 3' single cell version 2,Karolinska Institutet,external,braun_2023_brain,10X187_2_ABC_1
...,...,...,...,...,...,...,...,...
EGAN00002408195,EGAD00001006049,EGAN00002408195,10X89_6,NaN,Karolinska Institutet,external,braun_2023_brain,10X89_6
EGAN00002408193,EGAD00001006049,EGAN00002408193,10X89_8,NaN,Karolinska Institutet,external,braun_2023_brain,10X89_8
EGAN00002408187,EGAD00001006049,EGAN00002408187,10X96_2,NaN,Karolinska Institutet,external,braun_2023_brain,10X96_2


In [46]:
samples_v1.loc[~(samples_v1.library_acc.isin(samples.irods_name) | samples_v1.library_acc.isin(samples.author_sample_id)),'study'].value_counts()

Series([], Name: count, dtype: int64)

In [47]:
samples['HDCA_version'] = 'v2'
samples.loc[samples.irods_name.isin(samples_v1.library_acc) | samples.author_sample_id.isin(samples_v1.library_acc),'HDCA_version'] = 'v1'
samples['HDCA_version'].value_counts()

HDCA_version
v1    980
v2    884
Name: count, dtype: int64

In [48]:
samples.author_sample_id.value_counts().value_counts()

count
1    1864
Name: count, dtype: int64

In [49]:
samples.library_id.value_counts().value_counts()

count
1    1864
Name: count, dtype: int64

In [50]:
samples_v1.shape

(980, 10)

# Add extra info for sanger samples

In [51]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
sids = samples.author_sample_id
sanger_sample_info = utils.fetch_dbs.get_sample_info_from_mlw(secrets['mlw'],sids)
sanger_sample_info

,sample_name,sqsc_sample_id,supplier_name,donor_id,accession_number,study_name,sqsc_study_id,pipeline_id_lims
0,FCA_GND10375779,5854186,Hrv86-GON-0-SC-1,FCA_GND10375779,EGAN00003218778,Developmental Gonads,5888,Chromium single cell 5 prime
0,FCA_GND10375780,5854187,Hrv86-GON-0-SC-2,FCA_GND10375780,EGAN00003218779,Developmental Gonads,5888,Chromium single cell 5 prime
0,FCA_GND8047885,4171842,F81-GON-0-SC-F-45N-2,FCA_GND8047885,None,Developmental Gonads,5888,Chromium single cell
0,FCA_GND8103050,4205389,F83-GON-0-SC-F-45N-1,FCA_GND8103050,None,Developmental Gonads,5888,Chromium single cell 5 prime
0,FCA_GND8103053,4205392,F84-GON-0-SC-F-45N-3,FCA_GND8103053,None,Developmental Gonads,5888,Chromium single cell 5 prime
...,...,...,...,...,...,...,...,...
0,WSSS8062675,4179161,A43_TH_CD45N_5GEX_2,WSSS8062675,ERS3604657,WSSS_Research_Study_HCA,5857,Chromium single cell
0,WSSS8084742,4180204,C40_TH_TOT_1,WSSS8084742,ERS3630340,WSSS_Research_Study_HCA,5857,Chromium single cell
0,WSSS8084743,4180205,C40_TH_TOT_2,WSSS8084743,ERS3630341,WSSS_Research_Study_HCA,5857,Chromium single cell
0,WSSS8084744,4180206,C41_TH_TOT_1,WSSS8084744,ERS3630342,WSSS_Research_Study_HCA,5857,Chromium single cell


In [52]:
samples["found_internal"] = False
samples.loc[samples.author_sample_id.isin(sanger_sample_info.sample_name),"found_internal"] = True
sanger_not_sanger = samples[~samples["found_internal"] & (samples.institute=='Wellcome Sanger Institute')]
sanger_not_sanger.publication_id.value_counts()

publication_id
yayon_2024_thymus              39
suo_2022_immune_integration     5
Name: count, dtype: int64

In [53]:
samples['sanger_supplier_name'] = pd.NA
samples['sanger_10x_chemistry'] = pd.NA
d = dict(zip(sanger_sample_info.sample_name,sanger_sample_info.supplier_name))
samples.loc[samples["found_internal"],'sanger_supplier_name'] = [d[s] for s in samples.loc[samples["found_internal"],'author_sample_id']]
d = dict(zip(sanger_sample_info.sample_name,sanger_sample_info.pipeline_id_lims))
samples.loc[samples["found_internal"],'sanger_10x_chemistry'] = [d[s] for s in samples.loc[samples["found_internal"],'author_sample_id']]
samples

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name,HDCA_version,found_internal,sanger_supplier_name,sanger_10x_chemistry
library_id,,,,,,,,,,,,
GSM4968831,GSE162950,GSM4968831,aorta-gonad-mesonephros 4.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968831,v1,False,<NA>,<NA>
GSM4968832,GSE162950,GSM4968832,aorta-gonad-mesonephros 5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968832,v1,False,<NA>,<NA>
GSM4968833,GSE162950,GSM4968833,aorta-gonad-mesonephros 5.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968833,v1,False,<NA>,<NA>
GSM4968834,GSE162950,GSM4968834,aorta-gonad-mesonephros 6 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968834,v1,False,<NA>,<NA>
GSM4968835,GSE162950,GSM4968835,cord 4.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968835,v2,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...
GSM6068106,GSE201586,GSM6068106,humanDRG_sample14,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068106,v2,False,<NA>,<NA>
GSM6068107,GSE201586,GSM6068107,humanDRG_sample15,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068107,v2,False,<NA>,<NA>
GSM6068108,GSE201586,GSM6068108,humanDRG_sample16,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068108,v2,False,<NA>,<NA>


In [54]:
samples.sanger_10x_chemistry.value_counts(dropna=False)

sanger_10x_chemistry
<NA>                                  896
Chromium single cell                  428
Chromium single cell 5 prime          323
Chromium single cell 3 prime v3       128
Chromium single cell 3 prime HT v3     48
Chromium Visium                        21
Chromium single cell TCR               12
Chromium single cell 3 prime v2         8
Name: count, dtype: int64

In [55]:
samples[samples.sanger_10x_chemistry=='Chromium Visium'] # visium, will drop

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name,HDCA_version,found_internal,sanger_supplier_name,sanger_10x_chemistry
library_id,,,,,,,,,,,,
ERS20510955,PRJEB77091,ERS20510955,WSSS_F_IMMsp10864183,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20510955,v2,True,F136_IGLP4_3_THY,Chromium Visium
ERS20510966,PRJEB77091,ERS20510966,WSSS_F_IMMsp11604686,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20510966,v2,True,LP12_THY_F130_1,Chromium Visium
ERS20510968,PRJEB77091,ERS20510968,WSSS_F_IMMsp11604688,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20510968,v2,True,LP12_THY_F114_1,Chromium Visium
ERS20510969,PRJEB77091,ERS20510969,WSSS_F_IMMsp11604689,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20510969,v2,True,LP11_THY_F113_1,Chromium Visium
ERS20592940,PRJEB77091,ERS20592940,TA11486162,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20592940,v2,True,Z4-THY--FO-3-S20-B1,Chromium Visium
...,...,...,...,...,...,...,...,...,...,...,...,...
ERS20592939,PRJEB77091,ERS20592939,TA11486161,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20592939,v2,True,Z1-THY--FO-2-S23-A1,Chromium Visium
ERS20592941,PRJEB77091,ERS20592941,TA11486163,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20592941,v2,True,Z6-THY--FO-3-S20-C1,Chromium Visium
ERS20592942,PRJEB77091,ERS20592942,TA11486164,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20592942,v2,True,Z2-THY--FO-3-S18-D1,Chromium Visium


In [56]:
samples[samples.sanger_10x_chemistry=='Chromium single cell TCR'] # looks like TCR only, drop

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name,HDCA_version,found_internal,sanger_supplier_name,sanger_10x_chemistry
library_id,,,,,,,,,,,,
BHF_F_Hea10402917,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea10402917,BHF_F_Hea10402917,NaN,Wellcome Sanger Institute,internal,bayraktar_2024_heart,BHF_F_Hea10402917,v1,True,C86-HEA-0-SC-45P-1,Chromium single cell TCR
BHF_F_Hea11192323,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11192323,BHF_F_Hea11192323,NaN,Wellcome Sanger Institute,internal,bayraktar_2024_heart,BHF_F_Hea11192323,v1,True,C94-HEA-0-SC-45P-1,Chromium single cell TCR
BHF_F_Hea11192325,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11192325,BHF_F_Hea11192325,NaN,Wellcome Sanger Institute,internal,bayraktar_2024_heart,BHF_F_Hea11192325,v1,True,C97-HEA-0-SC-45P-1,Chromium single cell TCR
BHF_F_Hea11192327,Kanemaru_et_al_2024_bioRxiv,BHF_F_Hea11192327,BHF_F_Hea11192327,NaN,Wellcome Sanger Institute,internal,bayraktar_2024_heart,BHF_F_Hea11192327,v1,True,C99-HEA-0-SC-45P-1,Chromium single cell TCR
ERS11951382,E-MTAB-11343,ERS11951382,WSSS_F_Imm11192515,10x 5' v1,Wellcome Sanger Institute,external,suo_2022_immune_integration,ERS11951382,v2,True,F149-SPL-0-SC-FS-1,Chromium single cell TCR
ERS11951383,E-MTAB-11343,ERS11951383,WSSS_F_Imm11192516,10x 5' v1,Wellcome Sanger Institute,external,suo_2022_immune_integration,ERS11951383,v2,True,F149-SPL-0-SC-FS-2,Chromium single cell TCR
ERS11951384,E-MTAB-11343,ERS11951384,WSSS_F_Imm11192517,10x 5' v1,Wellcome Sanger Institute,external,suo_2022_immune_integration,ERS11951384,v2,True,F149-SPL-0-SC-FS-3,Chromium single cell TCR
ERS11951385,E-MTAB-11343,ERS11951385,WSSS_F_Imm11192518,10x 5' v1,Wellcome Sanger Institute,external,suo_2022_immune_integration,ERS11951385,v2,True,F149-SPL-0-SC-FS-4,Chromium single cell TCR
ERS20601046,PRJEB77091,ERS20601046,TA10211546,NaN,Wellcome Sanger Institute,external,yayon_2024_thymus,ERS20601046,v2,True,Z11-THY--SC-1,Chromium single cell TCR


In [57]:
samples = samples[~samples.sanger_10x_chemistry.isin(['Chromium single cell TCR','Chromium Visium'])]

# Check what is already on irods

In [58]:
samples['irods_path'] = '/archive/cellgeni/'
f = samples['source'] == 'external'
samples.loc[f,'irods_path'] = samples.loc[f,'irods_path'] + 'datasets/' + samples.loc[f,'dataset_acc'] + '/' + samples.loc[f,'irods_name']
f = samples['source'] == 'internal'
samples.loc[f,'irods_path'] = samples.loc[f,'irods_path'] + 'sanger/' + samples.loc[f,'irods_name']

/tmp/ipykernel_3007865/3059008109.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samples['irods_path'] = '/archive/cellgeni/'


In [59]:
samples['done'] = utils.fetch_dbs.irods_path_exists(samples['irods_path']+'/output/GeneFull/filtered/matrix.mtx.gz')
samples['done'].value_counts()

/tmp/ipykernel_3007865/3174734792.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samples['done'] = utils.fetch_dbs.irods_path_exists(samples['irods_path']+'/output/GeneFull/filtered/matrix.mtx.gz')


done
True     1709
False     122
Name: count, dtype: int64

In [60]:
samples.loc[~samples['done'],'irods_path'] = pd.NA

## Unfinished datasets

In [61]:
t=pd.crosstab(samples.publication_id,samples.irods_path.isna())
t[t[True]>0]

irods_path,False,True
publication_id,,
calvanese_2022_agm,20,4
hu_2025_amnion,3,1
lu_2024_drg,18,68
quach_2024_lung,63,39
xu_2023_embryo,118,7
yayon_2024_thymus,124,3


In [62]:
t=pd.crosstab(samples.dataset_acc,samples.irods_path.isna())
status = dict(zip(datasets.accession,datasets.status))
t['status'] = t.index
t['status'] = t['status'].replace(status)
t[(t[True]==0) & (t.status !='done')]

irods_path,False,True,status
dataset_acc,,,


In [63]:
t[t[True]>0]

irods_path,False,True,status
dataset_acc,,,
GSE135202,3,4,incomplete
GSE147520,4,3,incomplete
GSE157329,104,7,incomplete
GSE215895,0,39,cannot do
GSE245310,0,68,failed
GSE260715,3,1,incomplete


In [64]:
samples[(samples.dataset_acc=='GSE260715') & samples.irods_path.isna()]

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name,HDCA_version,found_internal,sanger_supplier_name,sanger_10x_chemistry,irods_path,done
library_id,,,,,,,,,,,,,,
GSM8122928,GSE260715,GSM8122928,CS19,NaN,California Institute of Technology,external,hu_2025_amnion,GSM8122928,v2,False,<NA>,<NA>,<NA>,False


In [65]:
samples

,dataset_acc,library_id,author_sample_id,library_type,institute,source,publication_id,irods_name,HDCA_version,found_internal,sanger_supplier_name,sanger_10x_chemistry,irods_path,done
library_id,,,,,,,,,,,,,,
GSM4968831,GSE162950,GSM4968831,aorta-gonad-mesonephros 4.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968831,v1,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE162950/GSM4968831,True
GSM4968832,GSE162950,GSM4968832,aorta-gonad-mesonephros 5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968832,v1,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE162950/GSM4968832,True
GSM4968833,GSE162950,GSM4968833,aorta-gonad-mesonephros 5.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968833,v1,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE162950/GSM4968833,True
GSM4968834,GSE162950,GSM4968834,aorta-gonad-mesonephros 6 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968834,v1,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE162950/GSM4968834,True
GSM4968835,GSE162950,GSM4968835,cord 4.5 week,NaN,"Broad Stem Cell Research Center, UCLA",external,calvanese_2022_agm,GSM4968835,v2,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE162950/GSM4968835,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM6068106,GSE201586,GSM6068106,humanDRG_sample14,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068106,v2,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE201586/GSM6068106,True
GSM6068107,GSE201586,GSM6068107,humanDRG_sample15,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068107,v2,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE201586/GSM6068107,True
GSM6068108,GSE201586,GSM6068108,humanDRG_sample16,NaN,"Institute of Biophysics, Chinese Academy of Sc...",external,lu_2024_drg,GSM6068108,v2,False,<NA>,<NA>,/archive/cellgeni/datasets/GSE201586/GSM6068108,True


## Save libraries

In [66]:
today = date.today().strftime("%y%m%d")
samples[['publication_id','institute', 'source','dataset_acc','library_id','author_sample_id','sanger_10x_chemistry','irods_path','HDCA_version']].to_csv('./libraries_lists/hdca_v2_libraries'+today+'.csv', index=False)

In [67]:
pd.crosstab(samples.HDCA_version,samples.irods_path.isna())

irods_path,False,True
HDCA_version,,
v1,976,0
v2,733,122


## Internal samples to reprocess

In [53]:
f = (~samples['done']) & (samples['source'] == 'internal')
f.value_counts()

False    1876
Name: count, dtype: int64

In [55]:
t=pd.crosstab(samples.dataset_acc,f)
t

col_0,False
dataset_acc,
E-MTAB-10187,16
E-MTAB-10551,37
E-MTAB-10552,12
E-MTAB-11278,43
E-MTAB-11343,83
...,...
Kanemaru_et_al_2024_bioRxiv,23
PRJEB37165,22
PRJEB41514,12


In [239]:
sids = pd.DataFrame({'sample':samples.loc[f,'irods_name']})
sids.to_csv('../sanger_samples2.csv',index=False)

In [243]:
samples.loc[samples.dataset_acc=='E-MTAB-9536','done']


library_id
ERS23679606    False
ERS23679607    False
ERS23679608    False
ERS23679609    False
ERS23679610    False
               ...  
ERS23679627    False
ERS23679628    False
ERS23679629    False
ERS23679630    False
ERS23679631    False
Name: done, Length: 26, dtype: bool